In [1]:
# ==========================================================
# DECISION TREE EXPERIMENT
# Manual Calculations + Iris Implementation + Extensions
# ==========================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_boston
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error

# ==========================================================
# PART A: MANUAL CALCULATIONS
# ==========================================================

print("="*60)
print("PART A: MANUAL CALCULATIONS")
print("="*60)

# Small dataset
data = [
    [0,0,0],[0,0,0],[1,0,0],[2,0,0],
    [2,1,1],[2,1,1],[1,1,1],[0,1,0],
    [2,0,1],[0,1,1]
]

df = pd.DataFrame(data, columns=['Age','Income','Buy'])
print("\nSample Dataset:\n", df)

# ---- Gini Impurity ----
def gini_impurity(y):
    p1 = sum(y)/len(y)
    p0 = 1 - p1
    return 1 - (p1**2 + p0**2)

labels = df['Buy'].values
gini_root = gini_impurity(labels)

print("\nGini Impurity (Root):", round(gini_root,4))

# ---- Entropy ----
def entropy_calc(y):
    p1 = sum(y)/len(y)
    p0 = 1 - p1
    entropy = 0
    if p1 > 0:
        entropy -= p1*np.log2(p1)
    if p0 > 0:
        entropy -= p0*np.log2(p0)
    return entropy

entropy_root = entropy_calc(labels)
print("Entropy (Root):", round(entropy_root,4))

# ---- Manual Split ----
left = df[df['Age']<=1]['Buy'].values
right = df[df['Age']>1]['Buy'].values

gini_left = gini_impurity(left)
gini_right = gini_impurity(right)

weighted_gini = (len(left)/len(labels))*gini_left + (len(right)/len(labels))*gini_right
information_gain = gini_root - weighted_gini

print("\nGini after split:", round(weighted_gini,4))
print("Information Gain:", round(information_gain,4))

# ==========================================================
# PART B: IRIS DATASET IMPLEMENTATION
# ==========================================================

print("\n" + "="*60)
print("PART B: IRIS DATASET IMPLEMENTATION")
print("="*60)

iris = load_iris()
X, y = iris.data, iris.target
feature_names = iris.feature_names
class_names = iris.target_names

X_train, X_test, y_train, y_test = train_test_split(
    X,y,test_size=0.3,random_state=42)

# ---- Gini Tree ----
dt_gini = DecisionTreeClassifier(criterion='gini',max_depth=3,random_state=42)
dt_gini.fit(X_train,y_train)

# ---- Entropy Tree ----
dt_entropy = DecisionTreeClassifier(criterion='entropy',max_depth=3,random_state=42)
dt_entropy.fit(X_train,y_train)

# ---- Accuracy ----
acc_gini = accuracy_score(y_test,dt_gini.predict(X_test))
acc_entropy = accuracy_score(y_test,dt_entropy.predict(X_test))

print("\nGini Accuracy:", round(acc_gini,3))
print("Entropy Accuracy:", round(acc_entropy,3))

# ---- Visualize Trees ----
plt.figure(figsize=(14,6))
plt.subplot(1,2,1)
plot_tree(dt_gini, feature_names=feature_names,
          class_names=class_names, filled=True)
plt.title("Gini Tree")

plt.subplot(1,2,2)
plot_tree(dt_entropy, feature_names=feature_names,
          class_names=class_names, filled=True)
plt.title("Entropy Tree")
plt.show()

# ---- Feature Importance ----
print("\nFeature Importance (Gini):")
for name, imp in zip(feature_names, dt_gini.feature_importances_):
    print(name, ":", round(imp,3))

print("\nFeature Importance (Entropy):")
for name, imp in zip(feature_names, dt_entropy.feature_importances_):
    print(name, ":", round(imp,3))

# ---- Compare Different Depths ----
print("\nDepth Comparison:")
print("Depth | Gini | Entropy")
for depth in range(1,6):
    dt_g = DecisionTreeClassifier(criterion='gini',max_depth=depth)
    dt_e = DecisionTreeClassifier(criterion='entropy',max_depth=depth)
    dt_g.fit(X_train,y_train)
    dt_e.fit(X_train,y_train)
    acc_g = accuracy_score(y_test,dt_g.predict(X_test))
    acc_e = accuracy_score(y_test,dt_e.predict(X_test))
    print(f"{depth:5d} | {acc_g:.3f} | {acc_e:.3f}")

# ==========================================================
# EXTENSIONS
# ==========================================================

print("\n" + "="*60)
print("EXTENSIONS")
print("="*60)

# ---- Cost Complexity Pruning ----
path = dt_gini.cost_complexity_pruning_path(X_train,y_train)
ccp_alphas = path.ccp_alphas

dt_pruned = DecisionTreeClassifier(ccp_alpha=ccp_alphas[1])
dt_pruned.fit(X_train,y_train)
print("\nPruned Tree Accuracy:",
      accuracy_score(y_test,dt_pruned.predict(X_test)))

# ---- Function for Information Gain ----
def information_gain_calc(parent, left, right):
    total = len(parent)
    gain = entropy_calc(parent)
    gain -= (len(left)/total)*entropy_calc(left)
    gain -= (len(right)/total)*entropy_calc(right)
    return gain

print("\nManual Information Gain (Entropy):",
      round(information_gain_calc(labels,left,right),4))

# ---- Decision Tree Regressor ----
print("\nDecision Tree Regressor Example")

# Using synthetic regression data
from sklearn.datasets import make_regression
X_reg, y_reg = make_regression(n_samples=200,n_features=4,noise=10,random_state=42)

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg,y_reg,test_size=0.3,random_state=42)

dt_reg = DecisionTreeRegressor(max_depth=4)
dt_reg.fit(Xr_train,yr_train)

mse = mean_squared_error(yr_test,dt_reg.predict(Xr_test))
print("Regression MSE:", round(mse,3))

# ---- Export Decision Rules ----
print("\nDecision Rules (Gini Tree):\n")
rules = export_text(dt_gini, feature_names=feature_names)
print(rules)

print("\n================ EXPERIMENT COMPLETED =================")

ImportError: 
`load_boston` has been removed from scikit-learn since version 1.2.

The Boston housing prices dataset has an ethical problem: as
investigated in [1], the authors of this dataset engineered a
non-invertible variable "B" assuming that racial self-segregation had a
positive impact on house prices [2]. Furthermore the goal of the
research that led to the creation of this dataset was to study the
impact of air quality but it did not give adequate demonstration of the
validity of this assumption.

The scikit-learn maintainers therefore strongly discourage the use of
this dataset unless the purpose of the code is to study and educate
about ethical issues in data science and machine learning.

In this special case, you can fetch the dataset from the original
source::

    import pandas as pd
    import numpy as np

    data_url = "http://lib.stat.cmu.edu/datasets/boston"
    raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)
    data = np.hstack([raw_df.values[: